In [9]:
import numpy as np
import seaborn as sns
import pandas as pd

df = sns.load_dataset('titanic')

df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [10]:
df.drop(columns=['deck','alive'],inplace=True)

In [11]:
df=pd.get_dummies(df,columns=['sex','embarked','class','who','adult_male','embark_town','alone'],drop_first=True,dtype=int)

In [12]:
df.isnull().sum()

,0
survived,0
pclass,0
age,177
sibsp,0
parch,0
fare,0
sex_male,0
embarked_Q,0
embarked_S,0
class_Second,0


In [13]:
df['age'].fillna(df['age'].median(),inplace=True)

/tmp/ipykernel_3564/2790788646.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['age'].fillna(df['age'].median(),inplace=True)


In [14]:
df.head()

,survived,pclass,age,sibsp,parch,fare,sex_male,embarked_Q,embarked_S,class_Second,class_Third,who_man,who_woman,adult_male_True,embark_town_Queenstown,embark_town_Southampton,alone_True
0,0,3,22.0,1,0,7.2500,1,0,1,0,1,1,0,1,0,1,0
1,1,1,38.0,1,0,71.2833,0,0,0,0,0,0,1,0,0,0,0
2,1,3,26.0,0,0,7.9250,0,0,1,0,1,0,1,0,0,1,1
3,1,1,35.0,1,0,53.1000,0,0,1,0,0,0,1,0,0,1,0
4,0,3,35.0,0,0,8.0500,1,0,1,0,1,1,0,1,0,1,1


In [15]:
X=df.drop(columns=['survived'])
y=df['survived']

In [16]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [17]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [18]:
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input

results = []

for hidden_layers in range(1, 8):
    for nodes in range(3, 16):

        model = Sequential()
        model.add(Input(shape=(X_train.shape[1],)))

        # Create hidden layers
        for i in range(hidden_layers):
            model.add(Dense(nodes, activation='relu'))

        # Output layer
        model.add(Dense(1, activation='sigmoid'))

        # Compile model
        model.compile(
            optimizer='adam',
            loss='binary_crossentropy',
            metrics=['accuracy']
        )

        # Train model
        model.fit(
            X_train,
            y_train,
            epochs=50,
            batch_size=32,
            verbose=0
        )

        # Evaluate model
        loss, accuracy = model.evaluate(
            X_test,
            y_test,
            verbose=0
        )

        results.append({
            'Hidden Layers': hidden_layers,
            'Nodes': nodes,
            'Accuracy': accuracy
        })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='Accuracy',ascending=False)

In [19]:
results_df

,Hidden Layers,Nodes,Accuracy
5,1,8,0.837989
25,2,15,0.832402
33,3,10,0.832402
64,5,15,0.832402
51,4,15,0.832402
...,...,...,...
0,1,3,0.787709
40,4,4,0.776536
14,2,4,0.765363
26,3,3,0.586592
